<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о фактурах за поставленные товары или оказанные услуги. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.

Требования к базовому классу Invoice:

• Атрибуты: Номер фактуры (InvoiceNumber), Дата выдачи (IssueDate), Общая сумма (TotalAmount).

• Методы:

o CalculateTotal(): метод для расчета общей суммы по фактуре.

o AddLine(LineItem lineItem): метод для добавления позиции в фактуру.

o RemoveLine(LineItem lineItem): метод для удаления позиции из фактуры.

Требования к производным классам:

ТоварнаяФактура (GoodsInvoice): Должна содержать дополнительные атрибуты, такие как Дата поставки (SupplyDate). Метод AddLine() должен быть переопределен для добавления информации о дате поставки товара при добавлении позиции.
УслуговаяФактура (ServiceInvoice): Должна содержать дополнительные атрибуты, такие как Дата оказания услуги (ServiceDate). Метод RemoveLine() должен быть переопределен для добавления информации о причине аннулирования услуги при удалении позиции.
КомбинированнаяФактура (CombinedInvoice) (если требуется третий класс): Должна содержать дополнительные атрибуты, такие как Наличие возврата (ReturnAllowed). Метод CalculateTotal() должен быть переопределен для учета возможного возврата товара или услуги при расчете общей суммы.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

public delegate void InvoiceHandler(string message);

interface IDiscount
{
    double GetDiscount();
}

class LineItem
{
    public string Name { get; set; }
    public double Price { get; set; }
    public int Quantity { get; set; }
    public string Unit { get; set; }
    public string Category { get; set; }

    public LineItem(string name, double price, int quantity = 1)
    {
        Name = name;
        Price = price;
        Quantity = quantity;
        Unit = "шт.";
        Category = "Общее";
    }

    public double GetSum() => Price * Quantity;

    public void ChangeQuantity(int quantity)
    {
        if (quantity >= 0) Quantity = quantity;
    }

    public void DisplayInfo()
    {
        Console.WriteLine($"{Name}: {Quantity} {Unit}, цена: {Price:F2}, сумма: {GetSum():F2}");
    }
}

class Invoice
{
    private int invoiceNumber;
    private DateTime issueDate;
    private double totalAmount;
    private string customer;

    protected LineItem[] lines = new LineItem[10];
    protected int count = 0;

    public int InvoiceNumber { get { return invoiceNumber; } set { invoiceNumber = value; } }
    public DateTime IssueDate { get { return issueDate; } set { issueDate = value; } }
    public double TotalAmount { get { return totalAmount; } protected set { totalAmount = value; } }
    public string Customer { get { return customer; } set { customer = value; } }
    public string Status { get; set; }
    public string Notes { get; set; }

    public string Currency { get; set; }
    public bool IsPaid { get; set; }
    public string CreatedBy { get; set; }

    public event InvoiceHandler InvoiceChanged;

    public Invoice(int number, DateTime date)
    {
        InvoiceNumber = number;
        IssueDate = date;
        Status = "Создана";
        Notes = "";
        Customer = "";
        Currency = "RUB";
        IsPaid = false;
        CreatedBy = "Система";
    }

    protected void OnInvoiceChanged(string message)
    {
        InvoiceChanged?.Invoke(message);
    }

    public void MarkAsPaid()
    {
        IsPaid = true;
        Status = "Оплачена";
        OnInvoiceChanged($"Фактура №{InvoiceNumber} оплачена");
    }

    public void ChangeCurrency(string currency)
    {
        Currency = currency;
    }

    public void SetCreatedBy(string user)
    {
        CreatedBy = user;
    }

    public virtual void CalculateTotal()
    {
        TotalAmount = 0;
        for (int i = 0; i < count; i++)
            if (lines[i] != null) TotalAmount += lines[i].GetSum();
    }

    public virtual void CalculateTotal(double discountPercent)
    {
        CalculateTotal();
        TotalAmount -= TotalAmount * discountPercent / 100;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        if (lineItem == null) return;
        if (count >= lines.Length) { Console.WriteLine("Массив позиций заполнен."); return; }
        lines[count] = lineItem; count++;
    }

    public virtual void AddLine(params LineItem[] items)
    {
        foreach (var item in items) AddLine(item);
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (lineItem == null) return;
        for (int i = 0; i < count; i++)
        {
            if (lines[i] == lineItem)
            {
                for (int j = i; j < count - 1; j++) lines[j] = lines[j + 1];
                lines[count - 1] = null; count--; break;
            }
        }
    }

    public int GetLineCount() { return count; }
    public void SetStatus(string status) { Status = status; }
    public void SetCustomer(string customer) { Customer = customer; }

    public virtual void DisplayInfo()
    {
        Console.WriteLine($"Фактура №{InvoiceNumber}, клиент: {Customer}, сумма: {TotalAmount:F2} {Currency}, статус: {Status}, оплачена: {IsPaid}, создал: {CreatedBy}");
    }
}

class GoodsInvoice : Invoice, IDiscount
{
    private DateTime supplyDate;
    private double discount;

    public DateTime SupplyDate { get { return supplyDate; } set { supplyDate = value; } }
    public double Discount { get { return discount; } set { discount = value; } }
    public string Supplier { get; set; }
    public string DeliveryAddress { get; set; }

    public string WarehouseCode { get; set; }
    public bool IsDelivered { get; set; }
    public double DeliveryCost { get; set; }

    public GoodsInvoice(int number, DateTime issueDate, DateTime supplyDate, double discount)
        : base(number, issueDate)
    {
        SupplyDate = supplyDate; Discount = discount;
        Supplier = ""; DeliveryAddress = "";
        WarehouseCode = "WH-01"; IsDelivered = false; DeliveryCost = 0;
    }

    public void ConfirmDelivery()
    {
        IsDelivered = true;
        OnInvoiceChanged($"Товарная фактура №{InvoiceNumber}: доставка подтверждена");
    }

    public void SetWarehouse(string code) { WarehouseCode = code; }

    public void SetDeliveryCost(double cost)
    {
        if (cost >= 0) DeliveryCost = cost;
    }

    public override void AddLine(LineItem lineItem)
    {
        base.AddLine(lineItem);
        Console.WriteLine("Товар добавлен.");
    }

    public override void CalculateTotal()
    {
        base.CalculateTotal();
        TotalAmount -= TotalAmount * Discount / 100;
        TotalAmount += DeliveryCost;
    }

    public override void CalculateTotal(double discountPercent)
    {
        CalculateTotal();
        TotalAmount -= TotalAmount * discountPercent / 100;
    }

    public void SetSupplier(string supplier) { Supplier = supplier; }
    public void ShowDeliveryDate() { Console.WriteLine("Дата поставки: " + SupplyDate.ToShortDateString()); }
    double IDiscount.GetDiscount() { return Discount; }

    public override void DisplayInfo()
    {
        Console.WriteLine($"Товарная фактура №{InvoiceNumber}, поставка: {SupplyDate:dd.MM.yyyy}, скидка: {Discount}%, сумма: {TotalAmount:F2}, склад: {WarehouseCode}, доставлено: {IsDelivered}");
    }
}

class ServiceInvoice : Invoice, IDiscount
{
    private DateTime serviceDate;
    private string cancelReason;
    private double discount;

    public DateTime ServiceDate { get { return serviceDate; } set { serviceDate = value; } }
    public string CancelReason { get { return cancelReason; } set { cancelReason = value; } }
    public double Discount { get { return discount; } set { discount = value; } }
    public string ServiceType { get; set; }
    public string Executor { get; set; }

    public int WarrantyMonths { get; set; }
    public bool IsCompleted { get; set; }
    public string ContractRef { get; set; }

    public ServiceInvoice(int number, DateTime issueDate, DateTime serviceDate, string cancelReason, double discount)
        : base(number, issueDate)
    {
        ServiceDate = serviceDate; CancelReason = cancelReason; Discount = discount;
        ServiceType = "Общая"; Executor = "";
        WarrantyMonths = 0; IsCompleted = false; ContractRef = "";
    }

    public void CompleteService()
    {
        IsCompleted = true;
        OnInvoiceChanged($"Счёт за услуги №{InvoiceNumber}: услуга завершена");
    }

    public void SetWarranty(int months) { WarrantyMonths = months; }

    public void SetContract(string contractRef) { ContractRef = contractRef; }

    public override void RemoveLine(LineItem lineItem)
    {
        base.RemoveLine(lineItem);
        Console.WriteLine("Операция удаления выполнена. Причина: " + CancelReason);
    }

    public void RemoveLine(string name)
    {
        for (int i = 0; i < count; i++)
            if (lines[i] != null && lines[i].Name == name) { RemoveLine(lines[i]); return; }
    }

    public void SetExecutor(string executor) { Executor = executor; }
    public void ShowServiceDate() { Console.WriteLine("Дата оказания услуги: " + ServiceDate.ToShortDateString()); }
    double IDiscount.GetDiscount() { return Discount; }

    public override void CalculateTotal()
    {
        base.CalculateTotal();
        TotalAmount -= TotalAmount * Discount / 100;
    }

    public override void CalculateTotal(double discountPercent)
    {
        CalculateTotal();
        TotalAmount -= TotalAmount * discountPercent / 100;
    }

    public override void DisplayInfo()
    {
        Console.WriteLine($"Счёт за услуги №{InvoiceNumber}, дата: {ServiceDate:dd.MM.yyyy}, скидка: {Discount}%, сумма: {TotalAmount:F2}, выполнено: {IsCompleted}, гарантия: {WarrantyMonths} мес.");
    }
}

class CombinedInvoice : Invoice
{
    private bool returnAllowed;
    private double returnAmount;

    public bool ReturnAllowed { get { return returnAllowed; } set { returnAllowed = value; } }
    public double ReturnAmount { get { return returnAmount; } set { returnAmount = value; } }
    public string ContractNumber { get; set; }
    public string PaymentMethod { get; set; }

    public bool IsMixedPayment { get; set; }
    public string PartnerName { get; set; }
    public int BonusPoints { get; set; }

    public CombinedInvoice(int number, DateTime issueDate, bool returnAllowed)
        : base(number, issueDate)
    {
        ReturnAllowed = returnAllowed; ReturnAmount = 0;
        ContractNumber = ""; PaymentMethod = "Наличные";
        IsMixedPayment = false; PartnerName = ""; BonusPoints = 0;
    }

    public void AddBonusPoints(int points)
    {
        if (points > 0)
        {
            BonusPoints += points;
            OnInvoiceChanged($"Фактура №{InvoiceNumber}: начислено {points} бонусов");
        }
    }

    public void SetPartner(string partner) { PartnerName = partner; }

    public void EnableMixedPayment() { IsMixedPayment = true; }

    public override void CalculateTotal()
    {
        base.CalculateTotal();
        if (ReturnAllowed)
        {
            TotalAmount -= ReturnAmount;
            Console.WriteLine($"Учтён возврат: {ReturnAmount:F2}");
        }
    }

    public void SetReturnAmount(double amount) { if (amount >= 0) ReturnAmount = amount; }
    public void ShowPaymentMethod() { Console.WriteLine("Способ оплаты: " + PaymentMethod); }

    public override void DisplayInfo()
    {
        Console.WriteLine($"Комбинированная фактура №{InvoiceNumber}, возврат: {(ReturnAllowed ? "разрешён" : "запрещён")}, сумма: {TotalAmount:F2}, партнёр: {PartnerName}, бонусы: {BonusPoints}");
    }
}

class InvoiceCollection<T> where T : Invoice
{
    private List<T> invoices;
    public InvoiceCollection(List<T> invoices)
    {
        if (invoices == null) throw new ArgumentNullException("invoices");
        this.invoices = invoices;
    }
    public void Add(T invoice) { if (invoice != null) invoices.Add(invoice); }
    public void Remove(T invoice) { if (invoice != null) invoices.Remove(invoice); }
    public T FindByNumber(int number)
    {
        foreach (T invoice in invoices)
            if (invoice != null && invoice.InvoiceNumber == number) return invoice;
        return null;
    }
    public void DisplayAll()
    {
        foreach (T invoice in invoices)
        {
            if (invoice == null) continue;
            invoice.CalculateTotal();
            invoice.DisplayInfo();
            Console.WriteLine();
        }
    }
}

static void OnInvoiceChanged(string message) => Console.WriteLine("[СОБЫТИЕ] " + message);

List<Invoice> list = new List<Invoice>();
InvoiceCollection<Invoice> collection = new InvoiceCollection<Invoice>(list);
Dictionary<int, Invoice> invoiceById = new Dictionary<int, Invoice>();

GoodsInvoice goods = new GoodsInvoice(1, DateTime.Now, DateTime.Now.AddDays(3), 5);
goods.SetCustomer("ООО Ромашка");
goods.SetSupplier("ООО Поставщик");
goods.AddLine(new LineItem("Стол", 5000, 2));
goods.AddLine(new LineItem("Стул", 1500, 4));
goods.SetWarehouse("WH-77");
goods.SetDeliveryCost(500);
goods.InvoiceChanged += OnInvoiceChanged;

ServiceInvoice service = new ServiceInvoice(2, DateTime.Now, DateTime.Now, "Отказ клиента", 10);
service.SetCustomer("ИП Иванов");
service.SetExecutor("Петров");
service.AddLine(new LineItem("Доставка", 800));
service.AddLine(new LineItem("Установка", 2000));
service.SetWarranty(12);
service.InvoiceChanged += OnInvoiceChanged;

CombinedInvoice combined = new CombinedInvoice(3, DateTime.Now, true);
combined.SetCustomer("ООО Вектор");
combined.AddLine(new LineItem("Ноутбук", 60000));
combined.AddLine(new LineItem("Мышка", 1200));
combined.SetReturnAmount(1200);
combined.SetPartner("ООО Партнёр");
combined.InvoiceChanged += OnInvoiceChanged;

collection.Add(goods);
collection.Add(service);
collection.Add(combined);

invoiceById[goods.InvoiceNumber] = goods;
invoiceById[service.InvoiceNumber] = service;
invoiceById[combined.InvoiceNumber] = combined;

collection.DisplayAll();

Action<Invoice> show = inv => inv.DisplayInfo();
show(goods);

var bigInvoices = list.Where(i => i.TotalAmount > 3000);
foreach (var inv in bigInvoices)
    Console.WriteLine($"Крупная фактура №{inv.InvoiceNumber}: {inv.TotalAmount:F2}");

goods.ConfirmDelivery();
service.CompleteService();
combined.AddBonusPoints(100);
goods.MarkAsPaid();

if (invoiceById.TryGetValue(2, out Invoice found))
    Console.WriteLine($"Найдено в Dictionary: №{found.InvoiceNumber}, клиент {found.Customer}");

Товар добавлен.
Товар добавлен.
Товарная фактура №1, поставка: 13.11.2026, скидка: 5%, сумма: 15700,00, склад: WH-77, доставлено: False

Счёт за услуги №2, дата: 10.11.2026, скидка: 10%, сумма: 2520,00, выполнено: False, гарантия: 12 мес.

Учтён возврат: 1200,00
Комбинированная фактура №3, возврат: разрешён, сумма: 60000,00, партнёр: ООО Партнёр, бонусы: 0

Товарная фактура №1, поставка: 13.11.2026, скидка: 5%, сумма: 15700,00, склад: WH-77, доставлено: False
Крупная фактура №1: 15700,00
Крупная фактура №3: 60000,00
[СОБЫТИЕ] Товарная фактура №1: доставка подтверждена
[СОБЫТИЕ] Счёт за услуги №2: услуга завершена
[СОБЫТИЕ] Фактура №3: начислено 100 бонусов
[СОБЫТИЕ] Фактура №1 оплачена
Найдено в Dictionary: №2, клиент ИП Иванов
